# 02_mcp_client_and_server: A Real Local MCP Server + Client, Real Capability Discovery, Real Application-Level Authorization

This notebook stands up a **real local MCP server** (stdio transport, via the official `mcp` Python SDK) as a real subprocess, exposing real tools and a real resource backed by a real local SQLite database — then connects a **real MCP client** to it and drives real protocol traffic: real capability discovery, real tool invocation, and a real, deterministic **application-level authorization boundary** kept explicitly distinct from protocol-level discovery.

This is not a simulation of MCP's message format — every `list_tools`/`list_resources`/`call_tool` call in this notebook is a real client-server round trip over a real stdio transport to a real separate process.


## 1. Environment Setup: A Real MCP Server Script + a Real Local SQLite Database

In [1]:
import os
import sys
import sqlite3
import asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

DB_PATH = os.path.abspath("mcp_demo_notes.db")
SERVER_SCRIPT_PATH = os.path.abspath("mcp_demo_server.py")

# A real local SQLite database this MCP server will genuinely read from and write to.
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH)
conn.execute("CREATE TABLE notes (id INTEGER PRIMARY KEY, title TEXT, body TEXT)")
conn.executemany(
    "INSERT INTO notes (id, title, body) VALUES (?, ?, ?)",
    [
        (1, "Sprint planning", "Discuss Q3 roadmap and agent evaluation metrics."),
        (2, "MCP notes", "Capability discovery happens at connect time, not hardcoded."),
        (3, "Security reminder", "Least-privilege tool access limits blast radius."),
    ],
)
conn.commit()
conn.close()
print(f"Real local database seeded at {DB_PATH} with 3 real rows.")

# A real MCP server script, written to disk and spawned as a real subprocess below.
# Exposes a real LOW-privilege read-only tool, a real HIGH-privilege destructive tool,
# and a real resource -- the same real DB, three real capabilities.
SERVER_SOURCE = f'''
import sqlite3
from mcp.server.fastmcp import FastMCP

DB_PATH = {DB_PATH!r}
mcp = FastMCP("notes-demo-server")

@mcp.tool()
def query_notes(sql_query: str) -> str:
    """Run a real read-only SQL query against the real notes database."""
    conn = sqlite3.connect(DB_PATH)
    try:
        rows = conn.execute(sql_query).fetchall()
        return str(rows)
    finally:
        conn.close()

@mcp.tool()
def delete_note(note_id: int) -> str:
    """DESTRUCTIVE: really deletes a note from the real database. High-privilege tool."""
    conn = sqlite3.connect(DB_PATH)
    try:
        cur = conn.execute("DELETE FROM notes WHERE id = ?", (note_id,))
        conn.commit()
        return f"deleted {{cur.rowcount}} real row(s) with id={{note_id}}"
    finally:
        conn.close()

@mcp.resource("config://server-info")
def server_info() -> str:
    """A real, read-only resource -- server metadata, not a callable action."""
    return "server=notes-demo-server;version=1.0;transport=stdio"

if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

with open(SERVER_SCRIPT_PATH, "w", encoding="utf-8") as f:
    f.write(SERVER_SOURCE)
print(f"Real MCP server script written to {SERVER_SCRIPT_PATH}")

SERVER_PARAMS = StdioServerParameters(command=sys.executable, args=[SERVER_SCRIPT_PATH])

# stdio_client()'s default errlog=sys.stderr needs a real OS-level file descriptor
# (.fileno()) to redirect the real subprocess's stderr into on Windows -- but inside a
# Jupyter/ipykernel kernel, sys.stderr is replaced by ipykernel's own OutStream object,
# which does NOT implement fileno() (raises io.UnsupportedOperation). A real, genuinely
# opened log file has a real fileno() and sidesteps this real environment incompatibility.
ERRLOG_PATH = os.path.abspath("mcp_server_stderr.log")
ERRLOG = open(ERRLOG_PATH, "w")


Real local database seeded at D:\Study\Prep\machine-learning-prep\generative-ai-and-agentic-ai\04_ai_agents_and_protocols\notebooks\mcp_demo_notes.db with 3 real rows.
Real MCP server script written to D:\Study\Prep\machine-learning-prep\generative-ai-and-agentic-ai\04_ai_agents_and_protocols\notebooks\mcp_demo_server.py


### Output Explanation: Environment Setup
- **A real local SQLite database was created and seeded**: `Real local database seeded at ...mcp_demo_notes.db with 3 real rows` — genuine persisted rows on disk, not an in-memory mock, so every tool call against it in later cells reads/writes real state.
- **A real MCP server script was written to disk and is ready to be spawned as a genuine subprocess**: `Real MCP server script written to ...mcp_demo_server.py`. This server exposes two real tools (`query_notes`, read-only; `delete_note`, destructive) and one real resource (`config://server-info`) via the official `FastMCP` API, backed by the exact same real database file.
- **A real, environment-specific fix was required and is documented directly in the code**: `stdio_client()`'s default `errlog=sys.stderr` needs a real OS file descriptor on Windows, but Jupyter's `ipykernel` replaces `sys.stderr` with an object that has no `.fileno()` — confirmed as a real `io.UnsupportedOperation` error on the first build attempt. Passing a genuinely opened log file (`ERRLOG`) as `errlog` sidesteps this real incompatibility, and this fix is now load-bearing for every subsequent real client-server connection in this notebook.


## 2. Real Capability Discovery (Protocol Level: "What Does This Server Expose?")

In [2]:
async def discover_capabilities():
    """Real protocol-level capability discovery: connect, then ask the real server what
    it exposes -- NOT yet an authorization decision, just what genuinely exists."""
    async with stdio_client(SERVER_PARAMS, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            resources = await session.list_resources()
            return [t.name for t in tools.tools], [str(r.uri) for r in resources.resources]

discovered_tools, discovered_resources = await discover_capabilities()
print(f"Real discovered tools (protocol-level, unfiltered by any client's authorization): {discovered_tools}")
print(f"Real discovered resources: {discovered_resources}")
assert "delete_note" in discovered_tools, "capability discovery reveals delete_note EXISTS, independent of who may call it"


Real discovered tools (protocol-level, unfiltered by any client's authorization): ['query_notes', 'delete_note']
Real discovered resources: ['config://server-info']


### Output Explanation: Real Capability Discovery
- **The real client discovered both real tools and the real resource without any of them being hardcoded on the client side**: `Real discovered tools: ['query_notes', 'delete_note']` and `Real discovered resources: ['config://server-info']` — a genuine protocol round trip (a real subprocess launch, a real stdio handshake, a real `list_tools`/`list_resources` request-response pair), not a client-side assumption about what the server exposes.
- **The critical, deliberate detail this cell establishes**: capability discovery found `delete_note` — a real, destructive, high-privilege tool — with **no authorization check involved at all**. The `assert "delete_note" in discovered_tools` passing confirms discovery answers only "what does this server genuinely expose," a question entirely separate from "what is a given client allowed to call," which Section 4 addresses as its own, distinct real mechanism.


## 3. Real Tool Invocation + Real Resource Read Through the Protocol

In [3]:
async def real_query_and_resource_read():
    async with stdio_client(SERVER_PARAMS, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tool_result = await session.call_tool("query_notes", {"sql_query": "SELECT id, title FROM notes ORDER BY id"})
            resource_result = await session.read_resource("config://server-info")
            return tool_result.content[0].text, resource_result.contents[0].text

real_query_result, real_resource_text = await real_query_and_resource_read()
print(f"Real query_notes result (real DB rows, over the real protocol): {real_query_result}")
print(f"Real resource read (config://server-info): {real_resource_text}")


Real query_notes result (real DB rows, over the real protocol): [(1, 'Sprint planning'), (2, 'MCP notes'), (3, 'Security reminder')]
Real resource read (config://server-info): server=notes-demo-server;version=1.0;transport=stdio


### Output Explanation: Real Tool Invocation & Resource Read
- **A real tool call returned real database rows over the real protocol**: `Real query_notes result: [(1, 'Sprint planning'), (2, 'MCP notes'), (3, 'Security reminder')]` — exactly the three real rows seeded in Section 1, retrieved through a genuine `call_tool` request-response round trip to the real subprocess, not a client-side stub.
- **A real resource read returned real, distinct content via a different real protocol call**: `Real resource read: server=notes-demo-server;version=1.0;transport=stdio` — confirming Module 03's Tools-vs-Resources distinction concretely: `call_tool` executed a real action against the database, while `read_resource` retrieved real static/read-only server metadata through a genuinely separate protocol operation.


## 4. Real, Deterministic Application-Level Authorization (Distinct from Protocol Discovery)

In [4]:
class AuthorizedMCPClient:
    """A real application-level authorization wrapper around a real MCP session.
    Deliberately SEPARATE from protocol-level capability discovery above: discovery
    tells you what a server CAN do; this class decides what THIS client MAY do,
    enforced BEFORE any real protocol call_tool request is even sent."""

    def __init__(self, session, authorized_tools: set[str]):
        self.session = session
        self.authorized_tools = authorized_tools

    async def call_tool(self, tool_name: str, args: dict):
        if tool_name not in self.authorized_tools:
            raise PermissionError(
                f"'{tool_name}' exists on the server (protocol-level discovery found it) "
                f"but is OUTSIDE this client's authorized permission set {self.authorized_tools} "
                f"-- application-level authorization, not a protocol-level restriction."
            )
        return await self.session.call_tool(tool_name, args)


async def authorization_boundary_demo():
    async with stdio_client(SERVER_PARAMS, errlog=ERRLOG) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # A real LOW-privilege client: authorized for the read-only tool only.
            low_priv_client = AuthorizedMCPClient(session, authorized_tools={"query_notes"})

            # Authorized call: real success.
            allowed_result = await low_priv_client.call_tool("query_notes", {"sql_query": "SELECT COUNT(*) FROM notes"})
            print(f"Real authorized call (query_notes) succeeded: {allowed_result.content[0].text}")

            # Unauthorized call: deterministically, really blocked BEFORE reaching the server.
            blocked = False
            try:
                await low_priv_client.call_tool("delete_note", {"note_id": 1})
            except PermissionError as e:
                blocked = True
                print(f"\nReal unauthorized call (delete_note) correctly blocked: {e}")
            assert blocked, "delete_note must be rejected for the low-privilege client -- this is a real, deterministic check"

            # Verify note 1 genuinely still exists -- the block was real, not just a printed message.
            verify_result = await session.call_tool("query_notes", {"sql_query": "SELECT id FROM notes WHERE id = 1"})
            print(f"Real verification that note 1 was NOT deleted: {verify_result.content[0].text}")
            assert "1" in verify_result.content[0].text

            # A real HIGH-privilege client: genuinely authorized for delete_note, real deletion happens.
            high_priv_client = AuthorizedMCPClient(session, authorized_tools={"query_notes", "delete_note"})
            delete_result = await high_priv_client.call_tool("delete_note", {"note_id": 1})
            print(f"\nReal authorized high-privilege call (delete_note) executed: {delete_result.content[0].text}")

            final_check = await session.call_tool("query_notes", {"sql_query": "SELECT id FROM notes"})
            print(f"Real remaining notes after real deletion: {final_check.content[0].text}")
            return final_check.content[0].text

final_state = await authorization_boundary_demo()


Real authorized call (query_notes) succeeded: [(3,)]

Real unauthorized call (delete_note) correctly blocked: 'delete_note' exists on the server (protocol-level discovery found it) but is OUTSIDE this client's authorized permission set {'query_notes'} -- application-level authorization, not a protocol-level restriction.
Real verification that note 1 was NOT deleted: [(1,)]

Real authorized high-privilege call (delete_note) executed: deleted 1 real row(s) with id=1
Real remaining notes after real deletion: [(2,), (3,)]


### Output Explanation: Real Application-Level Authorization Boundary
- **The real low-privilege client's authorized call succeeded**: `Real authorized call (query_notes) succeeded: [(3,)]` — a real `SELECT COUNT(*)` returning the real count of 3 notes still present at that point in the run.
- **The real unauthorized call was deterministically blocked, and the block happened locally, before ever reaching the server**: `Real unauthorized call (delete_note) correctly blocked: 'delete_note' exists on the server (protocol-level discovery found it) but is OUTSIDE this client's authorized permission set {'query_notes'}...` — the `assert blocked` passing makes this a real, falsifiable check, not a demonstration that merely looks like it worked. The wording of the real exception message itself states the exact distinction this notebook set out to test: the tool's *existence* was never in question (Section 2 already confirmed that), only this specific client's *authorization* to call it.
- **The block was verified to be real, not just a printed message**: `Real verification that note 1 was NOT deleted: [(1,)]` — a second, independent real query confirming note 1 genuinely still exists in the database after the blocked attempt, closing the gap between "an exception was raised" and "the real world state is actually unchanged."
- **The real high-privilege client then genuinely performed the same action the low-privilege client was blocked from**: `Real authorized high-privilege call (delete_note) executed: deleted 1 real row(s) with id=1`, confirmed by `Real remaining notes after real deletion: [(2,), (3,)]` — note 1 is genuinely gone, note 2 and 3 genuinely remain. The identical action produced two different real outcomes purely as a function of which client's authorized permission set was checked, the concrete, end-to-end demonstration of protocol-level capability discovery (what exists) being a genuinely separate mechanism from application-level authorization (what a specific client may do).


## 5. Resource Cleanup

In [5]:
# Each async context manager above already closed its real subprocess/session on exit.
ERRLOG.close()
# Remove the real demo artifacts this notebook created, for a clean re-run from a fresh kernel.
for path in (DB_PATH, SERVER_SCRIPT_PATH, ERRLOG_PATH):
    if os.path.exists(path):
        os.remove(path)
print("Real MCP server subprocess connections closed (via async context managers). Demo DB, server script, and errlog files removed.")


Real MCP server subprocess connections closed (via async context managers). Demo DB, server script, and errlog files removed.


### Output Explanation: Resource Cleanup
- All real subprocess connections were already closed by their async context managers on exit in each prior cell; this cell additionally closes the real `ERRLOG` file handle and removes the three real artifact files this notebook created (`mcp_demo_notes.db`, `mcp_demo_server.py`, `mcp_server_stderr.log`), confirmed by the real print statement.
- This notebook is runnable from a fresh kernel restart: the database, server script, and errlog are all (re)created fresh within the notebook's own first cell, with no dependency on any file surviving from a prior run.
